# Capítulo 6 — Ecuaciones diferenciales como lenguaje del cambio

**Cuaderno interactivo de *La servilleta y el ordenador*.**

Cada sección reproduce una figura del capítulo. La gracia no es ejecutarlas: es **cambiar los parámetros y comprobar si ocurre lo que esperabas**.

> Antes de ejecutar cada celda, escribe en una línea qué esperas ver. Después mira si ocurrió. Y después, por qué.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / ''))
sys.path.insert(0, '../../../herramientas')
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / 'herramientas'))

import numpy as np
import matplotlib.pyplot as plt
from estilo_libro import C, use_style, rng, save

use_style()
%matplotlib inline

---

## Cuatro ecuaciones explican medio mundo. ¿Cuáles y por qué?

Relajación, crecimiento, saturación y oscilación: para cada una, la línea de
fases (dx/dt frente a x) y la solución temporal.

La figura responde: ¿qué te dice la línea de fases que no te dice la solución?

Ejecutar:  python fig_cuatro_modelos.py

*(script original: `codigo/fig_cuatro_modelos.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

MODELOS = [
    ("Relajación\n$\\dot x = -x/\\tau$",
     lambda x: -x / 2.0, np.linspace(-2, 2, 200), [2.0, 1.0, -1.5], (0, 10)),
    ("Crecimiento\n$\\dot x = rx$",
     lambda x: 0.4 * x, np.linspace(-0.5, 3, 200), [0.2, 0.5, 1.0], (0, 6)),
    ("Saturación (logística)\n$\\dot x = rx(1-x/K)$",
     lambda x: 0.8 * x * (1 - x / 1.0), np.linspace(-0.15, 1.4, 200),
     [0.02, 0.4, 1.3], (0, 14)),
    ("Oscilación\n$\\ddot x = -\\omega^2 x$", None, None, None, (0, 14)),
]

fig, axes = plt.subplots(2, 4, figsize=(12.0, 6.0),
                         gridspec_kw={"height_ratios": [1, 1.15],
                                      "wspace": 0.42, "hspace": 0.45})

for col, (titulo, f, malla, inicios, trango) in enumerate(MODELOS):
    ax_f, ax_t = axes[0, col], axes[1, col]

    if f is not None:
        # --- línea de fases -----------------------------------------------
        ax_f.plot(malla, f(malla), color=C.blue, lw=2)
        ax_f.axhline(0, color=C.ink, lw=1.0)
        # puntos fijos y flechas de flujo
        v = f(malla)
        h = 3 * (malla[1] - malla[0])          # margen mayor que el paso de malla
        cruces = malla[:-1][np.sign(v[:-1]) != np.sign(v[1:])]
        for xc in cruces:
            # estable si el flujo apunta hacia el punto fijo por ambos lados
            estable = f(xc + h) < 0 < f(xc - h)
            ax_f.plot(xc, 0, "o", ms=8, color=C.ink if estable else "white",
                      mec=C.ink, mew=1.6, zorder=5)
        for xm in malla[::28]:
            if abs(f(xm)) > 1e-3:
                ax_f.annotate("", xy=(xm + 0.16 * np.sign(f(xm)), 0),
                              xytext=(xm, 0),
                              arrowprops=dict(arrowstyle="->", color=C.red, lw=1.2))
        ax_f.set_xlabel("$x$"), ax_f.set_ylabel(r"$\dot x$")
        ax_f.set_title(titulo, fontsize=9.5)

        # --- soluciones ---------------------------------------------------
        for x0 in inicios:
            sol = solve_ivp(lambda t, y: [f(y[0])], trango, [x0],
                            dense_output=True, rtol=1e-8)
            tt = np.linspace(*trango, 300)
            ax_t.plot(tt, sol.sol(tt)[0], lw=1.8)
        ax_t.set_xlabel("$t$"), ax_t.set_ylabel("$x$")
    else:
        # --- el oscilador necesita dos dimensiones -------------------------
        w = 1.2
        for x0 in (0.4, 0.8, 1.2):
            th = np.linspace(0, 2 * np.pi, 200)
            ax_f.plot(x0 * np.cos(th), -x0 * w * np.sin(th), lw=1.6)
        ax_f.plot(0, 0, "o", ms=7, color="white", mec=C.ink, mew=1.6)
        ax_f.set_xlabel("$x$"), ax_f.set_ylabel(r"$\dot x$")
        ax_f.set_title(titulo, fontsize=9.5)
        ax_f.set_aspect("equal")
        tt = np.linspace(*trango, 400)
        for x0 in (0.4, 0.8, 1.2):
            ax_t.plot(tt, x0 * np.cos(w * tt), lw=1.6)
        ax_t.set_xlabel("$t$"), ax_t.set_ylabel("$x$")

fig.suptitle("Arriba: línea de fases (círculo lleno = estable, hueco = "
             "inestable).  Abajo: solución temporal.",
             fontsize=9.5, color=C.grey, y=1.0)
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## Cuando un sistema tiene dos relojes, ¿cuál manda?

Sistema de dos compartimentos con constantes muy distintas. Se muestra la
solución completa y las dos aproximaciones: la rápida y la lenta.

La figura responde: ¿qué significa «separación de escalas» y cuándo se puede
eliminar la variable rápida?

Ejecutar:  python fig_escalas_temporales.py

*(script original: `codigo/fig_escalas_temporales.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

K_RAPIDA, K_LENTA = 30.0, 0.5      # s^-1


def sistema(_t, y):
    """A -> B (rápida) -> C (lenta)."""
    a, b, _c = y
    return [-K_RAPIDA * a, K_RAPIDA * a - K_LENTA * b, K_LENTA * b]


sol = solve_ivp(sistema, (0, 12), [1.0, 0.0, 0.0], dense_output=True,
                rtol=1e-10, atol=1e-12)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.2, 4.1))

for ax, (t0, t1), titulo in [
        (ax1, (0, 0.35), "Escala rápida: $1/k_1 = 0{,}033$ s"),
        (ax2, (0, 12), "Escala lenta: $1/k_2 = 2$ s")]:
    tt = np.linspace(t0, t1, 600)
    y = sol.sol(tt)
    for i, (nombre, color) in enumerate([("A", C.red), ("B", C.blue),
                                         ("C", C.green)]):
        ax.plot(tt, y[i], color=color, lw=2, label=nombre)
    ax.set_xlabel("tiempo (s)"), ax.set_ylabel("concentración")
    ax.set_title(titulo, fontsize=10)
    ax.legend(fontsize=8.5)

ax1.axvline(1 / K_RAPIDA, color=C.grey, ls=":", lw=1.2)
ax1.text(1 / K_RAPIDA * 1.15, 0.6, r"$t=1/k_1$", fontsize=8.4, color=C.grey)
ax2.plot(np.linspace(0, 12, 300), 1 - np.exp(-K_LENTA * np.linspace(0, 12, 300)),
         "--", color=C.ink, lw=1.4,
         label="aproximación:\nB decae solo")
ax2.legend(fontsize=8)
ax2.annotate("en esta escala, A ya no existe:\nse puede eliminar del modelo",
             xy=(1.0, 0.05), xytext=(3.2, 0.35), fontsize=8.6, color=C.red,
             arrowprops=dict(arrowstyle="->", color=C.red, lw=1.0))

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## Dos ecuaciones acopladas: ¿por qué oscilan los linces y las liebres?

Modelo de Lotka-Volterra: series temporales, retrato de fases y la cantidad
conservada que explica las órbitas cerradas.

La figura responde: ¿por qué las oscilaciones no se amortiguan, y qué le pasa
al sistema si matas depredadores?

Ejecutar:  python fig_lotka_volterra.py

*(script original: `codigo/fig_lotka_volterra.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

ALFA, BETA, GAMMA, DELTA = 1.1, 0.4, 0.4, 0.1


def lv(_t, y):
    presa, depredador = y
    return [ALFA * presa - BETA * presa * depredador,
            DELTA * presa * depredador - GAMMA * depredador]


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.2, 4.2))

sol = solve_ivp(lv, (0, 60), [10, 5], dense_output=True, rtol=1e-10)
tt = np.linspace(0, 60, 2000)
y = sol.sol(tt)
ax1.plot(tt, y[0], color=C.green, lw=2, label="presas")
ax1.plot(tt, y[1], color=C.red, lw=2, label="depredadores")
ax1.set_xlabel("tiempo"), ax1.set_ylabel("población")
ax1.set_title("Las presas van por delante; los depredadores, detrás")
ax1.legend(fontsize=8.5)

for x0 in [6, 10, 16, 24]:
    s = solve_ivp(lv, (0, 60), [x0, 5], dense_output=True, rtol=1e-10)
    ts = np.linspace(0, 60, 3000)
    ax2.plot(*s.sol(ts), lw=1.5)
ax2.plot(GAMMA / DELTA, ALFA / BETA, "o", color=C.ink, ms=7, zorder=5)
ax2.annotate("punto fijo\n(coexistencia)", (GAMMA / DELTA, ALFA / BETA),
             textcoords="offset points", xytext=(10, 6), fontsize=8.4)
ax2.set_xlabel("presas"), ax2.set_ylabel("depredadores")
ax2.set_title("Órbitas cerradas: hay una cantidad conservada")

# Campo de direcciones
X, Y = np.meshgrid(np.linspace(0.5, 30, 18), np.linspace(0.2, 9, 14))
U = ALFA * X - BETA * X * Y
V = DELTA * X * Y - GAMMA * Y
norma = np.hypot(U, V)
ax2.quiver(X, Y, U / norma, V / norma, color=C.grey, alpha=0.45,
           width=0.003, scale=38)

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Cuánto tarda un café en enfriarse, y qué es exactamente tau?

Ajusta la ley de Newton del enfriamiento a datos sintéticos con ruido realista
y muestra que en escala logarítmica es una recta cuya pendiente es -1/tau.

La figura responde: ¿por qué el tiempo característico no depende de la
temperatura inicial?

Ejecutar:  python fig_taza_cafe.py

*(script original: `codigo/fig_taza_cafe.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(60)

T_AMB, TAU = 21.0, 24.0          # °C y minutos
t = np.arange(0, 91, 5.0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.2, 4.1))

for T0, color in [(88.0, C.red), (65.0, C.ochre), (45.0, C.blue)]:
    T = T_AMB + (T0 - T_AMB) * np.exp(-t / TAU) + r.normal(0, 0.6, t.size)
    ax1.plot(t, T, "o", color=color, ms=4, label=f"$T_0$ = {T0:.0f} °C")

    def modelo(tt, T_amb, T0_, tau):
        return T_amb + (T0_ - T_amb) * np.exp(-tt / tau)

    popt, _ = curve_fit(modelo, t, T, p0=[20, T0, 20])
    tt = np.linspace(0, 90, 200)
    ax1.plot(tt, modelo(tt, *popt), color=color, lw=1.4, alpha=0.8)
    ax2.semilogy(t, np.maximum(T - popt[0], 1e-2), "o", color=color, ms=4)
    ax2.semilogy(tt, (popt[1] - popt[0]) * np.exp(-tt / popt[2]), color=color,
                 lw=1.4, alpha=0.8)
    print(f"T0={T0:5.1f}  ->  T_amb={popt[0]:5.2f}  tau={popt[2]:5.2f} min")

ax1.axhline(T_AMB, color=C.grey, ls="--", lw=1.1)
ax1.text(70, T_AMB + 1.2, "temperatura ambiente", fontsize=8.4, color=C.grey)
ax1.set_xlabel("tiempo (min)"), ax1.set_ylabel("temperatura (°C)")
ax1.set_title("Tres cafés distintos")
ax1.legend(fontsize=8)

ax2.set_xlabel("tiempo (min)")
ax2.set_ylabel(r"$T-T_{\mathrm{amb}}$ (°C)")
ax2.set_title(r"En escala log: tres rectas paralelas de pendiente $-1/\tau$")
ax2.annotate(r"la misma $\tau$ para los tres:"
             "\nel sistema olvida su condición inicial",
             xy=(60, 5), xytext=(8, 1.4), fontsize=8.6, color=C.ink,
             arrowprops=dict(arrowstyle="->", color=C.ink, lw=1.0))

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 
